In [3]:
import pandas as pd

df = pd.read_csv("data/Housing.csv")

df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


In [5]:
#ML models dont understand yes or no
#convert yes/no columns to 1/0
binary_cols = [
    'mainroad', 'guestroom', 'basement',
    'hotwaterheating', 'airconditioning', 'prefarea'
]

#yes means 1
#no means 0
for col in binary_cols:
    df[col]= df[col].map( {'yes':1, 'no':0} )


In [6]:
df

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,furnished
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,furnished
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,semi-furnished
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,furnished
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,1,0,1,0,0,2,0,unfurnished
541,1767150,2400,3,1,1,0,0,0,0,0,0,0,semi-furnished
542,1750000,3620,2,1,1,1,0,0,0,0,0,0,unfurnished
543,1750000,2910,3,1,1,0,0,0,0,0,0,0,furnished


In [7]:
#conept name one-hot encoding 
#one-hot encoding -> convert categorical (text) data into numeric(0/1)

#1,0 means semi_furnished
#0,1 means unfurnished
#0,0 means furnished
df= pd.get_dummies(df, columns=['furnishingstatus'], drop_first=True)

In [8]:
#True and False work same as 1 and 0
#True means 1 and False means 0
#No need to convert True and False
#ML treats True and False as 1 and 0
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,False,False
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,False,False
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,True,False
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,False,False
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,False,False


In [9]:
#axis 1 means column
#axis 2 means row

#Train-Test split (real data)

X= df.drop("price", axis=1)
y= df["price"]


In [10]:
#split data using train test split
from sklearn.model_selection import train_test_split

#80 percent train and 20 percent test
X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size= 0.2, random_state=42
)


In [11]:
#Check shapes

#436 rows for training
print(X_train.shape)

#109 rows for testing
print(X_test.shape)


(436, 13)
(109, 13)


In [12]:
#Train models

#Model num 1- Linear Regression
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

In [13]:
#Model num 2- RandomForest
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor()
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

In [14]:
#Find errors for both models
from sklearn.metrics import mean_absolute_error

#y_test (the 20% data of prices)
#lr_pred (20% predicted prices by linear regression)
#rf_pred (20% predicted prices by random forest)
lr_error = mean_absolute_error(y_test,lr_pred)
rf_error = mean_absolute_error(y_test,rf_pred)

#Lower numeric value error means better model
print("Linear Regression Error:", lr_error)
print("Random Forest Error:", rf_error)

Linear Regression Error: 970043.4039201637
Random Forest Error: 1038321.5831804281


In [ ]:
#lr_error and rf_error means average of errors
#actual price - predicted price for all rows
#then take average, MAE = average of(actual-predicted) differences

In [15]:
#Choose linear regression cause it performed better than Random forest

In [16]:
#Feature importance
import pandas as pd

#importance holds one number for each column in X 
#get feature importance (coefficients)
importance = lr_model.coef_

#create dataframe
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importance
})

#sort in ascending order
feature_importance = feature_importance.sort_values(by="Importance", ascending=False)

feature_importance

,Feature,Importance
2,bathrooms,1.094445e+06
8,airconditioning,7.914267e+05
7,hotwaterheating,6.846499e+05
10,prefarea,6.298906e+05
3,stories,4.074766e+05
6,basement,3.902512e+05
4,mainroad,3.679199e+05
5,guestroom,2.316100e+05
9,parking,2.248419e+05
1,bedrooms,7.677870e+04


In [17]:
# Importance (lr_model.coef_) represents the weight or impact of each feature (column) on the predicted price.
# After training, the model learns a mathematical relationship like:
# price = (w1 * area) + (w2 * bedrooms) + (w3 * bathrooms) + ...
# where each 'w' is the importance of that feature. These values are NOT sums or averages of the column,
# but learned weights that minimize prediction error across all rows. Each importance value tells us:
# “if this feature increases by 1 unit (keeping others constant), how much will the predicted price change.”
# For example, if bathrooms has importance ≈ 1,094,445, it means adding 1 bathroom increases price by ~10.9 lakh.
# Higher absolute value = stronger impact on price, lower value = weaker impact.
importance

array([ 2.35968805e+02,  7.67787016e+04,  1.09444479e+06,  4.07476595e+05,
        3.67919948e+05,  2.31610037e+05,  3.90251176e+05,  6.84649885e+05,
        7.91426736e+05,  2.24841913e+05,  6.29890565e+05, -1.26881818e+05,
       -4.13645062e+05])

In [18]:
#Price Range(Confidence Logic)

#example prediction (first test value)
sample = X_test.iloc[0:1]

pred_price = lr_model.predict(sample)[0]

#create range
lower = pred_price - lr_error
upper = pred_price + lr_error

print("Predicted Price:", pred_price)
print("Price Range:",lower,"to",upper)

Predicted Price: 5164653.900339675
Price Range: 4194610.496419512 to 6134697.304259839


In [19]:
#save final model
import pickle
pickle.dump(lr_model, open("model.pkl","wb"))